# HKI + FastAPI: HTTP-Layer Envelope Enforcement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h3nok/HKI/blob/main/notebooks/03_fastapi_middleware.ipynb)

This notebook shows how to enforce HKI at the HTTP boundary using `HkiMiddleware`.
The middleware:

1. Reads the `X-HKI-Envelope` header on every request
2. Validates the envelope (rejects missing, expired, global, wildcard, or malformed)
3. Binds the validated envelope to `request.state.hki` for downstream use
4. Returns 401/403 with a structured error body — never silently passes

**Time to complete:** ~10 minutes  
**Requirements:** Python 3.11+, FastAPI

In [ ]:
%pip install hki-runtime fastapi httpx -q

---
## Part 1 — Add HkiMiddleware to any FastAPI app

In [ ]:
import json, base64
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from hki_runtime.fastapi import HkiMiddleware

app = FastAPI()
app.add_middleware(HkiMiddleware)  # one line — that's the entire integration

@app.post("/search")
async def search(request: Request, body: dict):
    # The validated envelope is available on every request
    env = request.state.hki
    return {
        "active_domain": env.active_domain,
        "query": body.get("query"),
        "results": [f"result from {env.active_domain} knowledge base"],
    }

client = TestClient(app, raise_server_exceptions=False)

def make_envelope(domain: str) -> str:
    env = {
        "hki_version": "1.0",
        "envelope_id": f"env_{domain}_001",
        "org_id": "org_acme",
        "subject_id": "user_42",
        "active_domain": domain,
        "authorized_domains": [domain],
        "purpose": "retrieve",
        "risk_tier": "read-only",
        "policy_pack_id": f"{domain}@2026-05",
        "issued_at": 0,
        "expires_at": 9_999_999_999,
        "issuer": "gateway.acme.internal",
        "signature": "ed25519:placeholder",
    }
    return base64.b64encode(json.dumps(env).encode()).decode()

print("App ready")

In [ ]:
# Valid pharmacy request
response = client.post(
    "/search",
    json={"query": "return policy"},
    headers={"X-HKI-Envelope": make_envelope("pharmacy")},
)
print(f"Status: {response.status_code}")
print(f"Body:   {response.json()}")

---
## Part 2 — Requests that HkiMiddleware blocks

In [ ]:
import json, base64

def bad_envelope(overrides: dict) -> str:
    base = {
        "hki_version": "1.0",
        "envelope_id": "env_bad",
        "org_id": "org_acme",
        "subject_id": "user_42",
        "active_domain": "pharmacy",
        "authorized_domains": ["pharmacy"],
        "purpose": "retrieve",
        "risk_tier": "read-only",
        "policy_pack_id": "pharmacy@2026-05",
        "issued_at": 0,
        "expires_at": 9_999_999_999,
        "issuer": "gateway",
        "signature": "sig",
    }
    base.update(overrides)
    return base64.b64encode(json.dumps(base).encode()).decode()

cases = [
    ("No envelope header",      {}, None),
    ("global active_domain",    {"X-HKI-Envelope": bad_envelope({"active_domain": "global"})}, None),
    ("wildcard active_domain",  {"X-HKI-Envelope": bad_envelope({"active_domain": "*"})}, None),
    ("expired envelope",        {"X-HKI-Envelope": bad_envelope({"expires_at": 1})}, None),
    ("domain not in authorized",{"X-HKI-Envelope": bad_envelope({"active_domain": "travel"})}, None),
]

for label, headers, _ in cases:
    r = client.post("/search", json={"query": "test"}, headers=headers)
    print(f"  {label:<35s} → HTTP {r.status_code}")

---
## Part 3 — Using the envelope in route handlers

After `HkiMiddleware` validates the envelope, it is available at `request.state.hki`.
Use it to enforce artifact visibility in your retrieval logic.

In [ ]:
from fastapi import FastAPI, Request, HTTPException
from fastapi.testclient import TestClient
from hki_runtime.fastapi import HkiMiddleware
from hki_runtime import assert_artifact_visible

app2 = FastAPI()
app2.add_middleware(HkiMiddleware)

# Simulated document store with domain labels
DOCS = {
    "ph_001": {"domain": "pharmacy", "content": "Return policy: 30 days"},
    "tr_001": {"domain": "travel",   "content": "Hotel cancellation: 48h"},
}

@app2.get("/doc/{doc_id}")
async def get_doc(doc_id: str, request: Request):
    doc = DOCS.get(doc_id)
    if not doc:
        raise HTTPException(404, "not found")

    # Check visibility against the HKI envelope
    issue = assert_artifact_visible(request.state.hki, {
        "org_id": "org_acme",
        "domain": doc["domain"],
        "artifact_type": "document",
        "artifact_id": doc_id,
    })
    if issue:
        raise HTTPException(403, issue.message)

    return {"doc_id": doc_id, "content": doc["content"]}

client2 = TestClient(app2, raise_server_exceptions=False)

# Pharmacy agent reads pharmacy document — OK
r = client2.get("/doc/ph_001", headers={"X-HKI-Envelope": make_envelope("pharmacy")})
print(f"pharmacy reads ph_001 → {r.status_code}: {r.json()}")

# Pharmacy agent tries to read travel document — BLOCKED
r = client2.get("/doc/tr_001", headers={"X-HKI-Envelope": make_envelope("pharmacy")})
print(f"pharmacy reads tr_001 → {r.status_code}: {r.json()}")

---
## Summary

```python
# Full HKI FastAPI integration — two imports, one line
from hki_runtime.fastapi import HkiMiddleware
from hki_runtime import assert_artifact_visible

app.add_middleware(HkiMiddleware)

# In any route handler:
env = request.state.hki   # validated envelope, ready to use
```

| What the middleware does automatically | What you add per route |
|---|---|
| Reads `X-HKI-Envelope` header | `assert_artifact_visible(env, artifact)` |
| Validates: not expired, not global, not wildcard | `evaluate_gateway_target(env, tool)` |
| Returns 401/403 on any violation | `reject_conflicting_scope_argument(env, body)` |
| Binds envelope to `request.state.hki` | — |

### Next
- [04 — Threat demos](./04_threat_demos.ipynb): all 15 HKI threats with before/after code